This is a **CLEAN** but vivid version of codes (along with visualizations) 
for analyzing the experiment ***freight collaboration-chessboard*** results

In [40]:
from dataclasses import dataclass
from enum import Enum
import os
import sys
import pandas as pd
from pathlib import Path

# Use repo-relative path so it works on other machines
notebook_dir = Path.cwd() / "python" / "test"
if notebook_dir.exists() and str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import matsim_output_reader
# --- reload the module to reflect any changes made during development ---
import importlib
importlib.reload(matsim_output_reader)

<module 'matsim_output_reader' from 'f:\\gitRepository\\matsim-libs-2024\\python\\test\\matsim_output_reader.py'>

# Configuration

In [16]:
# Resolve analysis path relative to repo root
repo_root = notebook_dir.parents[1]  # .../matsim-libs-2024
anls_path = repo_root / "output" / "chessboardCarrierReceiverCollab"

class DepotLocation(Enum):
    INSIDE = 'center'
    OUTSIDE = 'left'

class ReceiverDistribution(Enum):
    DISPERSED = 'DISPERSED'
    CLUSTERED = 'CLUSTERED'

ALLOCATION_FACTOR = 0.8
#--- Penalty ---#
PENALTY_LIST = [0, 0.0003, 0.0008, 0.0014, 0.0028, 0.0056,
                0.0098, 0.014, 0.0167, 0.0222, 0.028]
PENALTY_LIST_SCALE = [round(x * 3600) for x in PENALTY_LIST]  # scale to avoid float precision issues
PENALTY_LIST_SCALE[-1] = 100 
PENALTY_DICT = {k: v for k, v in zip(PENALTY_LIST, PENALTY_LIST_SCALE)}
#--- Penalty ---#

TOTAL_INSTANCES = 50


# Test

In [41]:
test_folder = os.listdir(anls_path)[5]
print(f"Analyzing results from folder: {test_folder}")
test_carriers_df, test_shipments_df = matsim_output_reader.read_carriers(
    os.path.join(anls_path, test_folder, 'output_carriers.xml.gz')
)
test_receivers_dict = matsim_output_reader.read_receivers(
    os.path.join(anls_path, test_folder, 'receivers.xml.gz'), (6,8))

Analyzing results from folder: center-CLUSTERED-penSweep-af0.80-p0.0000-exactShapley-i05


In [19]:
test_carriers_df

,carrier_id,depot_link_ids,depot_link_id,vehicle_ids,num_vehicles,carrier_score,num_shipments
0,carrier1,"[i(5,5)R, i(5,5)R]","i(5,5)R","[lightVan1, heavyVan1]",2,808.972959,10


In [20]:
test_shipments_df

,carrier_id,shipment_id,pickup_link_id,delivery_link_id,size,start_pickup,end_pickup,start_delivery,end_delivery,pickup_service_time,delivery_service_time
0,carrier1,Orderreceiver_001,"i(5,5)R","i(6,6)",500,00:00:00,596523:14:07,06:00:00,10:00:00,00:00:00,00:20:00
1,carrier1,Orderreceiver_012,"i(5,5)R","i(4,6)",500,00:00:00,596523:14:07,06:00:00,10:00:00,00:00:00,00:20:00
2,carrier1,Orderreceiver_023,"i(5,5)R","j(3,7)",500,00:00:00,596523:14:07,06:00:00,08:00:00,00:00:00,00:20:00
3,carrier1,Orderreceiver_034,"i(5,5)R","j(5,6)",500,00:00:00,596523:14:07,06:00:00,11:00:00,00:00:00,00:20:00
4,carrier1,Orderreceiver_045,"i(5,5)R","j(3,6)",500,00:00:00,596523:14:07,06:00:00,12:00:00,00:00:00,00:20:00
5,carrier1,Orderreceiver_056,"i(5,5)R","i(5,5)R",500,00:00:00,596523:14:07,06:00:00,11:00:00,00:00:00,00:20:00
6,carrier1,Orderreceiver_067,"i(5,5)R","j(4,7)R",500,00:00:00,596523:14:07,06:00:00,10:00:00,00:00:00,00:20:00
7,carrier1,Orderreceiver_078,"i(5,5)R","j(4,6)R",500,00:00:00,596523:14:07,06:00:00,11:00:00,00:00:00,00:20:00
8,carrier1,Orderreceiver_089,"i(5,5)R","j(6,5)R",500,00:00:00,596523:14:07,06:00:00,12:00:00,00:00:00,00:20:00
9,carrier1,Orderreceiver_0910,"i(5,5)R","j(6,6)R",500,00:00:00,596523:14:07,06:00:00,13:00:00,00:00:00,00:20:00


In [42]:
test_receivers_dict

{'collaborative_receivers': [{'id': 'receiver_00',
   'time_window': (6, 10),
   'time_window_str': '06:00:00 - 10:00:00',
   'score': -74.17120658380937},
  {'id': 'receiver_01',
   'time_window': (6, 10),
   'time_window_str': '06:00:00 - 10:00:00',
   'score': -69.45360188555604},
  {'id': 'receiver_03',
   'time_window': (6, 11),
   'time_window_str': '06:00:00 - 11:00:00',
   'score': -61.84777044984135},
  {'id': 'receiver_04',
   'time_window': (6, 12),
   'time_window_str': '06:00:00 - 12:00:00',
   'score': -74.95195770015869},
  {'id': 'receiver_05',
   'time_window': (6, 11),
   'time_window_str': '06:00:00 - 11:00:00',
   'score': -59.796377211110936},
  {'id': 'receiver_06',
   'time_window': (6, 10),
   'time_window_str': '06:00:00 - 10:00:00',
   'score': -94.13923719587295},
  {'id': 'receiver_07',
   'time_window': (6, 11),
   'time_window_str': '06:00:00 - 11:00:00',
   'score': -80.6363005563495},
  {'id': 'receiver_08',
   'time_window': (6, 12),
   'time_window_str